In [19]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, T5ForConditionalGeneration, T5Tokenizer
import torch
import pandas as pd
import json

In [5]:
with open("Model_dataset/cv.json", "r") as file:
    CV_DATA= json.load(file)

In [21]:
q_type_models= [
    'model/fine_tuned_question_classifier_model_lite-default',
    'model/fine_tuned_question_classifier_model_lite-Adam',
    'model/fine_tuned_question_classifier_model_lite-AdamW',
    'model/fine_tuned_question_classifier_model_lite-SGD']
qa_type_model= 't5-large'

In [22]:
# QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[0])
# QUESTION_CLASSIFIER_MODEL.eval()
# QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[0])
# ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label

# QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_model)
# QUESTION_ANSWER_MODEL.eval()
# QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_model, legacy= False)


# Comparing results for all different optimiser result:

In [23]:
def get_dataframe_for_comparision(questions):
    table_data= {
        "questions": questions
    }
    for q_type_model in q_type_models:
        QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_model)
        QUESTION_CLASSIFIER_MODEL.eval()
        QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_model)
        ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label

        def get_question_type_predictions(texts):
            inputs = QUESTION_CLASSIFIER_TOKENIZER(texts, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
            logits = outputs.logits
            predicted_classes = torch.argmax(logits, dim=1)
            id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
            return [id2label[idx.item()] for idx in predicted_classes]
        table_data[q_type_model.split("-")[-1]]= get_question_type_predictions(questions)
    return pd.DataFrame(table_data)

In [ ]:
questions = [
    "What have you used for web development?",
    "How do I train a neural network?",
    "How much do you want to earn?",
    "What are you looking for in the new company?",
    "How much break do you need per day?",
    "Share you most recent salary draw.",
    "Rate yourself 1 to 10 for python developer.",
    "What is your most recent degree?",
    "Have you completed masters?",
    "Have you completed Bsc",
    "Who is the CEO of your company?",
    "Who inspire you the most?",
    "have you rechived any performance bounus?",
    "How frequent you expect performance review?",
    "Have you review others code before?"
    "What are the top framework you use?",
    "Have you completed any personal project recently?",
    "Have you a lead a team before?",
    "How many years of experience do you have with SQL?",
    "What you expect from the current company?"
]
get_dataframe_for_comparision(questions)

# Using classifier

In [25]:
QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[0])
QUESTION_CLASSIFIER_MODEL.eval()
QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[0])
ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label

In [26]:
def get_question_type_prediction(text):
    inputs = QUESTION_CLASSIFIER_TOKENIZER(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
    logits = outputs.logits
    predicted_classes = torch.argmax(logits, dim=1)
    id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
    return id2label[predicted_classes.item()]
# get_question_type_prediction("What have you used for web development?")

In [27]:
def get_question_type_predictions(texts):
    inputs = QUESTION_CLASSIFIER_TOKENIZER(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
    logits = outputs.logits
    predicted_classes = torch.argmax(logits, dim=1)
    id2label = QUESTION_CLASSIFIER_MODEL.config.id2label
    return [id2label[idx.item()] for idx in predicted_classes]

# Example usage
questions = [
    "What have you used for web development?",
    "How do I train a neural network?",
    "Who is the CEO of OpenAI?",
    "How much do you want to earn?",
    "What are you looking for in the new company?",
    "How much break do you need per day?",
    "Share you most recent salary draw.",
    "Rate yourself 1 to 10 for python developer.",
    "What is your most recent degree?",
    "Have you completed masters?",
    "Have you completed Bsc",
    "Who is the CEO of your company?",
    "Who inspire you the most?",
    "have you rechived any performance bounus?",
    "How frequent you expect performance review?"
]
# get_question_type_predictions(questions)

In [28]:
# def get_model_out_raw(question):
#     predicted_question_type= get_question_type_prediction(question)
#     context= CV_DATA[predicted_question_type]
#     input_text = f"question: {question} context: {context}"
#     inputs = QUESTION_ANSWER_TOKENIZER(input_text, return_tensors="pt")
#     outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)

#     return predicted_question_type, QUESTION_ANSWER_TOKENIZER.decode(outputs[0], skip_special_tokens=True)


# # get_model_out_raw("Total work experince in python?")